# First Meet with LLM
Connect to an LLM, send your first prompt, and use output parsers and prompt templates.

## 1. Install dependencies

In [ ]:
%pip install -q openai langchain langchain_openai langchain_core langchain_community python-dotenv

## 2. Load API key

Create a `.env` file in the same directory with:
```
OPENAI_API_KEY=sk-...
```
Or set the environment variable before running the notebook.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Add it to a .env file or export it."
print("API key loaded.")

## 3. Connect to OpenAI

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

response = llm.invoke("Hello!")
print(response)

## 4. StrOutputParser — get plain text back

By default `invoke` returns an `AIMessage` object. Pipe through `StrOutputParser` to get a plain string.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = llm | StrOutputParser()

result = chain.invoke("Hello!")
print(result)  # plain string

## 5. ChatPromptTemplate — reusable prompts with parameters

In [ ]:
import textwrap
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Tell me a short story about {subject}")

chain = prompt | llm | StrOutputParser()

story = chain.invoke({"subject": "cat"})
print(textwrap.fill(story, 80))

## 6. Token usage with get_openai_callback

In [ ]:
from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    story = chain.invoke({"subject": "dog"})

print(f"Tokens used:       {cb.total_tokens}")
print(f"Prompt tokens:     {cb.prompt_tokens}")
print(f"Completion tokens: {cb.completion_tokens}")
print(f"Cost (USD):        ${cb.total_cost:.6f}")
print()
print(textwrap.fill(story, 80))

## 7. EPAM DIAL connection (EPAM employees only)

Request your DIAL key at https://chat.lab.epam.com/ (requires EPAM VPN).  
Set `OPENAI_API_KEY` to your DIAL key in `.env`.

In [ ]:
from langchain_openai import AzureChatOpenAI

epam_dial = AzureChatOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    api_version="2023-07-01-preview",
    azure_endpoint="https://ai-proxy.lab.epam.com",
    model="gpt-4o-mini-2024-07-18",
    temperature=0.0,
)

epam_chain = epam_dial | StrOutputParser()
print(epam_chain.invoke("Hello!"))